# English Premier League data — collection phase

Data for the 20 clubs in the 2026/27 Premier League, as requested on October 6, 2026. This starts the EPL component of the September 30 NBA/EPL calibrated betting proposal. The notebook inspects collected data; no probabilities, returns or arbitrage claims have been fitted. Read `README.md` and `SOURCE_GUIDE.md` before constructing historical predictors.

In [1]:
from pathlib import Path
import pandas as pd

candidates = [Path.cwd(), Path.cwd() / 'final_project' / 'epl_data', Path.cwd() / 'epl_data']
ROOT = next(path for path in candidates if (path / 'metadata' / 'current_teams.csv').exists())
def load(source, table):
    return pd.read_parquet(ROOT / 'data' / 'processed' / source / f'{table}.parquet')
pd.set_option('display.max_columns', 25)

In [2]:
catalog = pd.read_csv(ROOT / 'metadata' / 'dataset_catalog.csv')
display(catalog[['source', 'table', 'rows', 'columns', 'season_min', 'season_max']])

,source,table,rows,columns,season_min,season_max
0,clubelo,rated_match_history,320,18,NaN,NaN
1,clubelo,rating_history,3896,8,NaN,NaN
2,clubelo,ratings_snapshot,20,4,NaN,NaN
3,espn,match_info,424,25,NaN,NaN
4,espn,match_key_events,5989,29,NaN,NaN
...,...,...,...,...,...,...
85,understat,shots,83455,28,2014-15,2026-27
86,understat,team_match_metrics,6180,32,2014-15,2026-27
87,understat,team_season_metrics,180,28,2014-15,2026-27
88,understat,xg_basis_comparison,6180,13,2014-15,2026-27


In [3]:
teams = pd.read_csv(ROOT / 'metadata' / 'team_identity_crosswalk.csv')
assert len(teams) == 20 and teams['team'].is_unique
assert not teams['team'].str.contains('West Ham|Burnley|Wolverhampton', case=False).any()
display(teams)

,team,fpl_team_id,fpl_team_code,fpl_name,short_name,season,as_of_date,membership_source,transfermarkt_club_id,espn_team_id,understat_team_id,sportsdb_team_id,clubelo_slug
0,Arsenal,1,3,Arsenal,ARS,2026-27,2026-10-06,https://www.premierleague.com/en/news/4673099/...,11,359,83,133604,Arsenal
1,Aston Villa,2,7,Aston Villa,AVL,2026-27,2026-10-06,https://www.premierleague.com/en/news/4673099/...,405,362,71,133601,AstonVilla
2,Bournemouth,3,91,Bournemouth,BOU,2026-27,2026-10-06,https://www.premierleague.com/en/news/4673099/...,989,349,73,134301,Bournemouth
3,Brentford,4,94,Brentford,BRE,2026-27,2026-10-06,https://www.premierleague.com/en/news/4673099/...,1148,337,244,134355,Brentford
4,Brighton & Hove Albion,5,36,Brighton,BHA,2026-27,2026-10-06,https://www.premierleague.com/en/news/4673099/...,1237,331,220,133619,Brighton
5,Chelsea,6,8,Chelsea,CHE,2026-27,2026-10-06,https://www.premierleague.com/en/news/4673099/...,631,363,80,133610,Chelsea
6,Coventry City,7,9,Coventry City,COV,2026-27,2026-10-06,https://www.premierleague.com/en/news/4673099/...,990,388,294,133625,Coventry
7,Crystal Palace,8,31,Crystal Palace,CRY,2026-27,2026-10-06,https://www.premierleague.com/en/news/4673099/...,873,384,78,133632,CrystalPalace
8,Everton,9,11,Everton,EVE,2026-27,2026-10-06,https://www.premierleague.com/en/news/4673099/...,29,368,72,133615,Everton
9,Fulham,10,54,Fulham,FUL,2026-27,2026-10-06,https://www.premierleague.com/en/news/4673099/...,931,370,228,133600,Fulham


## Current player and availability snapshots

The primary player master starts from all FPL-listed entries, including unavailable players. `can_select` identifies selectable entries. Transfermarkt links are conservative and unmatched players remain in the FPL universe. Current news, market values and injuries describe retrieval time.

In [4]:
players = load('integrated', 'current_player_master')
print(f'FPL-listed: {len(players):,}; selectable: {int(players.can_select.sum()):,}')
display(players.groupby('team').agg(listed=('fpl_player_id', 'size'), selectable=('can_select', 'sum')))
columns = ['team', 'player_name', 'position_name', 'can_select', 'status', 'news', 'minutes',
           'goals_scored', 'assists', 'expected_goals', 'expected_assists', 'tm_market_value_eur']
display(players[[column for column in columns if column in players]].head(20))

FPL-listed: 667; selectable: 562


,listed,selectable
team,,
Arsenal,29,24
Aston Villa,36,30
Bournemouth,29,27
Brentford,29,29
Brighton & Hove Albion,37,32
Chelsea,34,27
Coventry City,37,30
Crystal Palace,36,29
Everton,25,22


,team,player_name,position_name,can_select,status,news,minutes,goals_scored,assists,expected_goals,expected_assists,tm_market_value_eur
0,Arsenal,David Raya Martín,Goalkeeper,True,a,,450,0,0,0.00,0.01,30000000.0
1,Arsenal,Kepa Arrizabalaga Revuelta,Goalkeeper,True,a,,0,0,0,0.00,0.00,5000000.0
2,Arsenal,Illan Meslier,Goalkeeper,True,a,,0,0,0,0.00,0.00,8000000.0
3,Arsenal,Gabriel dos Santos Magalhães,Defender,True,a,,450,0,0,0.51,0.20,75000000.0
4,Arsenal,Jurriën Timber,Defender,True,a,,104,0,0,0.00,0.03,70000000.0
5,Arsenal,William Saliba,Defender,True,i,Back injury - Unknown return date,0,0,0,0.00,0.00,100000000.0
6,Arsenal,Myles Lewis-Skelly,Midfielder,True,a,,271,0,0,0.16,0.23,45000000.0
7,Arsenal,Riccardo Calafiori,Defender,True,a,,416,0,2,0.67,0.62,55000000.0
8,Arsenal,Piero Hincapié,Defender,True,a,,32,0,0,0.00,0.00,50000000.0
9,Arsenal,Benjamin White,Defender,True,d,Groin injury - 75% chance of playing,315,0,1,0.18,0.29,30000000.0


## Historical team matches and bookmaker odds

Use E0 for Premier League modeling. E1/E2/E3 retain lower-division history for these clubs. Source date coverage is measured from the downloaded records; missing values remain missing.

In [5]:
matches = load('football_data', 'matches_current_cohort')
epl = matches.loc[matches['division'].eq('E0')].copy()
display(epl.groupby('season').agg(matches=('source_match_key', 'size'), first=('match_date', 'min'), last=('match_date', 'max')))
odds = load('football_data', 'match_odds_1x2_long')
display(odds.head())

,matches,first,last
season,,,
1993-94,372,1993-08-14,1994-05-08
1994-95,406,1994-08-20,1995-05-14
1995-96,324,1995-08-19,1996-05-04
1996-97,324,1996-08-17,1997-05-11
1997-98,308,1997-08-09,1998-05-10
1998-99,308,1998-08-15,1999-05-16
1999-00,308,1999-08-07,2000-05-14
2000-01,338,2000-08-19,2001-05-19
2001-02,324,2001-08-18,2002-05-11


,source_match_key,season,division,match_date,HomeTeam,AwayTeam,home_team_current,away_team_current,completed_asof,bookmaker_or_aggregate,source_prefix,snapshot_type,quote_timestamp_utc,decimal_odds_home,source_column_home,decimal_odds_draw,source_column_draw,decimal_odds_away,source_column_away,valid_complete_1x2_quote,has_invalid_decimal_odds,is_market_aggregate,is_exchange,pinnacle_stale_warning
0,fd_1b7946da7f278b2e82f9be30,2024-25,E0,2024-08-16,Man United,Fulham,Manchester United,Fulham,True,1XBet,1XB,preclosing,None,1.68,1XBH,4.32,1XBD,5.03,1XBA,True,False,False,False,False
1,fd_1ac47248f28818fa9496c341,2024-25,E0,2024-08-17,Arsenal,Wolves,Arsenal,None,True,1XBet,1XB,preclosing,None,1.20,1XBH,7.65,1XBD,16.00,1XBA,True,False,False,False,False
2,fd_6801f3f4983333b52031a697,2024-25,E0,2024-08-17,Newcastle,Southampton,Newcastle United,None,True,1XBet,1XB,preclosing,None,1.37,1XBH,5.74,1XBD,8.10,1XBA,True,False,False,False,False
3,fd_8f087e2b5dbdc942adc1974d,2024-25,E0,2024-08-17,Everton,Brighton,Everton,Brighton & Hove Albion,True,1XBet,1XB,preclosing,None,2.68,1XBH,3.66,1XBD,2.63,1XBA,True,False,False,False,False
4,fd_956ea8101369c933ee78f986,2024-25,E0,2024-08-17,West Ham,Aston Villa,None,Aston Villa,True,1XBet,1XB,preclosing,None,2.42,1XBH,3.91,1XBD,2.81,1XBA,True,False,False,False,False


In [6]:
features = load('football_data', 'team_match_lagged_features')
known = features['features_last_match_date'].notna()
assert (features.loc[known, 'features_last_match_date'] < features.loc[known, 'match_date']).all()
display(features.head())

,source_match_key,team,opponent,season,division,match_date,venue,prior_observed_matches_all_divisions,features_last_match_date,previous_season_division,previous_season_matches,previous_season_win_percentage,...,season_prior_corners_difference_sum,season_prior_corners_difference_observations,season_prior_expected_goals_for_mean,season_prior_expected_goals_for_sum,season_prior_expected_goals_for_observations,season_prior_expected_goals_against_mean,season_prior_expected_goals_against_sum,season_prior_expected_goals_against_observations,season_prior_expected_goals_difference_mean,season_prior_expected_goals_difference_sum,season_prior_expected_goals_difference_observations,rest_days
0,fd_0de61b7733b62ffd982dd747,Arsenal,Coventry City,1993-94,E0,1993-08-14,home,0,None,None,NaN,NaN,...,NaN,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,NaN
1,fd_c36227a6d74186f160655ec2,Arsenal,Tottenham Hotspur,1993-94,E0,1993-08-16,away,1,1993-08-14,None,NaN,NaN,...,NaN,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,2.0
2,fd_206339d27bc0f4c945394c09,Arsenal,Sheffield Weds,1993-94,E0,1993-08-21,away,2,1993-08-16,None,NaN,NaN,...,NaN,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,5.0
3,fd_f7560be3cd79e46311003b71,Arsenal,Leeds United,1993-94,E0,1993-08-24,home,3,1993-08-21,None,NaN,NaN,...,NaN,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,3.0
4,fd_a6f89ac5ee9a110814bbda16,Arsenal,Everton,1993-94,E0,1993-08-28,home,4,1993-08-24,None,NaN,NaN,...,NaN,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,4.0


## Historical player fixtures

Historical club attribution follows the actual fixture, preserving transfers. The archive current-season GW1 overlaps official FPL; use official histories for 2026/27 when combining. Zero-minute fantasy records are preserved; filter `minutes > 0` for actual appearances. Match statistics must be lagged.

In [7]:
historical = load('fpl_archive', 'player_gameweeks')
current = load('fpl_current', 'player_match_history')
player_history = pd.concat([historical.loc[historical['season'].ne('2026-27')], current], ignore_index=True, sort=False)
assert not player_history.duplicated(['season', 'element', 'fixture']).any()
display(player_history.groupby('season').agg(records=('element', 'size'), players=('element', 'nunique'), clubs=('canonical_team', 'nunique')))

,records,players,clubs
season,,,
2016-17,13164,387,11
2017-18,12305,355,11
2018-19,12809,371,12
2019-20,13637,399,12
2020-21,16036,468,13
2021-22,16631,483,13
2022-23,21552,636,16
2023-24,22307,655,15
2024-25,22040,650,16


## Advanced expected-goals data and coverage

Different providers use different xG definitions. Full-season player summaries, detailed shots, lineups and historical values are descriptive records, not pre-match forecasts.

In [8]:
xg = load('understat', 'team_match_metrics')
display(xg.head())
display(pd.read_csv(ROOT / 'metadata' / 'quality_checks.csv'))

,h_a,xG,xGA,npxG,npxGA,ppda__att,ppda__def,ppda_allowed__att,ppda_allowed__def,deep,deep_allowed,scored,...,npxGD,season,season_start_year,team,source_team,understat_team_id,match_id,opponent_source_team,opponent_team,match_datetime_source,source,requested_as_of_date
0,a,0.909774,0.423368,0.909774,0.423368,323,23,132,32,4,3,1,...,0.486406,2014-15,2014,Aston Villa,Aston Villa,71,4752,Stoke,None,2014-08-16 15:00:00,understat,2026-10-06
1,h,0.507525,0.699295,0.507525,0.699295,326,21,180,21,4,7,0,...,-0.191770,2014-15,2014,Aston Villa,Aston Villa,71,4759,Newcastle United,Newcastle United,2014-08-23 12:45:00,understat,2026-10-06
2,h,0.639316,0.288880,0.639316,0.288880,366,13,278,24,6,7,2,...,0.350436,2014-15,2014,Aston Villa,Aston Villa,71,4776,Hull,Hull City,2014-08-31 13:30:00,understat,2026-10-06
3,a,0.701676,0.728097,0.701676,0.728097,486,9,91,14,1,5,1,...,-0.026421,2014-15,2014,Aston Villa,Aston Villa,71,4726,Liverpool,Liverpool,2014-09-13 17:30:00,understat,2026-10-06
4,h,0.649013,1.362240,0.649013,1.362240,531,12,170,22,0,7,0,...,-0.713227,2014-15,2014,Aston Villa,Aston Villa,71,4731,Arsenal,Arsenal,2014-09-20 15:00:00,understat,2026-10-06


,check,passed,detail
0,clubelo/rated_match_history: CSV readable,True,320 rows; 18 fields
1,clubelo/rated_match_history: typed Parquet ava...,True,data/processed/clubelo/rated_match_history.csv
2,clubelo/rated_match_history: focal clubs withi...,True,team
3,clubelo/rating_history: CSV readable,True,"3,896 rows; 8 fields"
4,clubelo/rating_history: typed Parquet available,True,data/processed/clubelo/rating_history.csv
...,...,...,...
238,understat_context/team_context_metrics: CSV re...,True,"6,552 rows; 17 fields"
239,understat_context/team_context_metrics: typed ...,True,data/processed/understat_context/team_context_...
240,understat_context/team_context_metrics: focal ...,True,team
241,Saved source SHA256 hashes,True,6522 verified; mismatches=[]


## Next modeling step

Choose past-season training/validation/test blocks, use only earlier completed match statistics, and compare calibration-driven versus accuracy-driven models on identical fixtures. Preserve the distinction between preclosing and closing odds. The collected static bookmaker snapshots do not establish synchronized executable arbitrage.